# 05 — Evaluation

Compare the trained U-Net model against the Douglas-Peucker baseline on the **test set** (Wageningen).

Metrics:
- **IoU** — overlap between predicted and true BRT coverage
- **Hausdorff distance** — worst-case boundary deviation
- **Polygon count ratio** — how well does the model match BRT polygon density
- **Vertex reduction** — how much geometry simplification occurred
- **Pixel accuracy** — raster-level classification accuracy

Run **03_baseline.ipynb** and **04_model.ipynb** first.


## 0. CONFIG

In [ ]:
from pathlib import Path

CONFIG = {
    "output_root":   Path("processed"),
    "model_dir":     Path("models"),
    "crs":           "EPSG:28992",

    # Must match 04_model.ipynb
    "image_size":    256,
    "unet_features": [32, 64, 128, 256],

    # Evaluate on this many test tiles (None = all)
    "eval_n_tiles":  None,

    # Vectorization: minimum polygon area to keep after
    # converting the predicted mask back to polygons (m²)
    "min_area_post_vectorize": 10.0,
}

## 1. Imports

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn

from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load encoders, model, and baseline results

In [ ]:
# Load encoders from JSON — no pickle, no class dependency
def load_encoder(path):
    with open(path) as f:
        data = json.load(f)
    class Encoder:
        pass
    enc = Encoder()
    enc.unknown_label = data["unknown_label"]
    enc.classes_      = data["classes"]
    enc.label_to_int  = data["label_to_int"]
    enc.int_to_label  = {int(k): v for k, v in data["int_to_label"].items()}
    enc._fitted       = True
    return enc

bgt_encoder = load_encoder(CONFIG["output_root"] / "bgt_encoder.json")
brt_encoder = load_encoder(CONFIG["output_root"] / "brt_encoder.json")

n_in  = len(bgt_encoder.classes_)
n_out = len(brt_encoder.classes_)

print(f"BGT classes: {n_in}")
print(f"BRT classes: {n_out}")

# Re-define UNet and ConvBlock here so this notebook is self-contained
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels, features):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.upconvs  = nn.ModuleList()
        ch = in_channels
        for feat in features:
            self.encoders.append(ConvBlock(ch, feat))
            self.pools.append(nn.MaxPool2d(2))
            ch = feat
        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)
        rev = features[::-1]
        ch = features[-1] * 2
        for feat in rev:
            self.upconvs.append(nn.ConvTranspose2d(ch, feat, 2, stride=2))
            self.decoders.append(ConvBlock(feat * 2, feat))
            ch = feat
        self.output_conv = nn.Conv2d(features[0], out_channels, 1)
    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.upconvs, self.decoders, skips[::-1]):
            x = up(x)
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = dec(x)
        return self.output_conv(x)

model = UNet(n_in, n_out, CONFIG["unet_features"]).to(DEVICE)
model.load_state_dict(torch.load(CONFIG["model_dir"] / "best_unet.pt", map_location=DEVICE))
model.eval()
print(f"Model loaded — {sum(p.numel() for p in model.parameters()):,} parameters")

# Load baseline results from notebook 03
baseline_df = pd.read_csv(CONFIG["output_root"] / "baseline_results.csv")
print(f"Baseline results: {len(baseline_df)} tiles")

## 3. Vectorization helper

Convert the predicted class mask back into a GeoDataFrame of polygons.

In [ ]:
def mask_to_geodataframe(mask: np.ndarray,
                          bounds: tuple,
                          brt_encoder,
                          min_area: float,
                          crs: str) -> gpd.GeoDataFrame:
    """
    Convert a (H, W) int label mask back to a GeoDataFrame of polygons.

    bounds : (minx, miny, maxx, maxy) in the target CRS
    min_area: small polygons below this area are dropped
    """
    from rasterio.features import shapes
    from rasterio.transform import from_bounds
    from shapely.geometry import shape

    minx, miny, maxx, maxy = bounds
    h, w = mask.shape
    transform = from_bounds(minx, miny, maxx, maxy, w, h)

    rows = []
    for geom_dict, class_idx in shapes(mask.astype(np.int32), transform=transform):
        if class_idx == 0:   # skip background
            continue
        geom = shape(geom_dict)
        if geom.area < min_area:
            continue
        label = brt_encoder.int_to_label.get(int(class_idx), "__UNKNOWN__")
        rows.append({"geometry": geom, "class_idx": int(class_idx), "class_label": label})

    if not rows:
        return gpd.GeoDataFrame(columns=["geometry", "class_idx", "class_label"], crs=crs)

    return gpd.GeoDataFrame(rows, crs=crs)


print("Vectorization helper defined")

## 4. Re-use metric functions from notebook 03

In [ ]:
# Copy of metric functions — keeping this notebook self-contained

def iou_polygon(a, b):
    try:
        inter = a.intersection(b).area
        union = a.union(b).area
        return inter / union if union > 0 else 0.0
    except Exception:
        return 0.0


def evaluate_tile(pred: gpd.GeoDataFrame, target: gpd.GeoDataFrame) -> dict:
    if len(pred) == 0 or len(target) == 0:
        return {"mean_iou": 0.0, "hausdorff": float("nan"),
                "count_ratio": 0.0, "vertex_reduction": float("nan")}

    pred_union   = pred.geometry.union_all()
    target_union = target.geometry.union_all()
    mean_iou     = iou_polygon(pred_union, target_union)

    try:
        hausdorff = pred_union.hausdorff_distance(target_union)
    except Exception:
        hausdorff = float("nan")

    count_ratio = len(pred) / max(len(target), 1)

    def total_verts(gdf):
        return sum(
            len(g.exterior.coords) if g.geom_type == "Polygon"
            else sum(len(p.exterior.coords) for p in g.geoms)
            for g in gdf.geometry
        )

    vr = 1.0 - (total_verts(pred) / (total_verts(target) + 1e-9))

    return {
        "mean_iou":         round(mean_iou, 4),
        "hausdorff":        round(hausdorff, 2),
        "count_ratio":      round(count_ratio, 3),
        "vertex_reduction": round(vr, 4),
    }


print("Metric functions defined")

## 5. Evaluate model on test set

In [ ]:
# Import rasterization functions from notebook 04 logic
# (Re-defined inline to keep this notebook self-contained)
from rasterio.features import rasterize as rio_rasterize
from rasterio.transform import from_bounds


def rasterize_bgt(gdf, class_col_encoded, n_classes, image_size):
    if len(gdf) == 0:
        return np.zeros((n_classes, image_size, image_size), dtype=np.float32)
    minx, miny, maxx, maxy = gdf.total_bounds
    if maxx == minx or maxy == miny:
        return np.zeros((n_classes, image_size, image_size), dtype=np.float32)
    transform = from_bounds(minx, miny, maxx, maxy, image_size, image_size)
    canvas = np.zeros((n_classes, image_size, image_size), dtype=np.float32)
    if class_col_encoded not in gdf.columns:
        return canvas
    for cls_idx in range(n_classes):
        sub = gdf[gdf[class_col_encoded] == cls_idx]
        shapes = [(g, 1) for g in sub.geometry if g and not g.is_empty]
        if shapes:
            canvas[cls_idx] = rio_rasterize(
                shapes, out_shape=(image_size, image_size),
                transform=transform, fill=0, dtype=np.float32
            )
    return canvas


with open(CONFIG["output_root"] / "tile_index.json") as f:
    tile_index = json.load(f)

test_tiles = tile_index["test"]
if CONFIG["eval_n_tiles"]:
    test_tiles = test_tiles[:CONFIG["eval_n_tiles"]]

print(f"Evaluating model on {len(test_tiles)} test tiles...")

model_results = []
pixel_correct = 0
pixel_total   = 0

bgt_enc_col = "plus_type_encoded"   # must match 01_data_prep
brt_enc_col = "typewegde_encoded"

for i, tile in enumerate(test_tiles):
    bgt = gpd.read_file(tile["bgt"])
    brt = gpd.read_file(tile["brt"])

    # Rasterize input
    x_arr = rasterize_bgt(bgt, bgt_enc_col, n_in, CONFIG["image_size"])
    x_t   = torch.from_numpy(x_arr).unsqueeze(0).to(DEVICE)

    # Predict
    with torch.no_grad():
        logits    = model(x_t)
        pred_mask = logits.argmax(dim=1).squeeze(0).cpu().numpy()

    # Pixel accuracy against rasterized BRT
    if brt_enc_col in brt.columns and len(brt) > 0:
        minx, miny, maxx, maxy = bgt.total_bounds
        t = from_bounds(minx, miny, maxx, maxy, CONFIG["image_size"], CONFIG["image_size"])
        brt_clipped = brt.cx[minx:maxx, miny:maxy]
        if len(brt_clipped) > 0:
            true_shapes = [
                (g, int(c))
                for g, c in zip(brt_clipped.geometry, brt_clipped[brt_enc_col])
                if g and not g.is_empty
            ]
            if true_shapes:
                true_mask = rio_rasterize(
                    true_shapes,
                    out_shape=(CONFIG["image_size"], CONFIG["image_size"]),
                    transform=t, fill=0, dtype=np.int32
                )
                pixel_correct += (pred_mask == true_mask).sum()
                pixel_total   += pred_mask.size

    # Vectorize prediction
    pred_gdf = mask_to_geodataframe(
        pred_mask, bgt.total_bounds, brt_encoder,
        CONFIG["min_area_post_vectorize"], CONFIG["crs"]
    )

    # Evaluate
    metrics = evaluate_tile(pred_gdf, brt)
    metrics["tile"] = tile["bgt"]
    metrics["city"] = tile["city"]
    model_results.append(metrics)

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_tiles)} tiles")

model_df = pd.DataFrame(model_results)
overall_pixel_acc = pixel_correct / max(pixel_total, 1)

print(f"\nDone. Pixel accuracy: {overall_pixel_acc*100:.2f}%")

## 6. Model vs Baseline comparison table

In [ ]:
metrics_cols = ["mean_iou", "hausdorff", "count_ratio", "vertex_reduction"]

baseline_means = baseline_df[metrics_cols].mean()
model_means    = model_df[metrics_cols].mean()

comparison = pd.DataFrame({
    "Baseline (DP)": baseline_means,
    "U-Net model":   model_means,
}).round(4)

# Mark improvement
# For IoU: higher is better. For Hausdorff: lower is better.
better = {
    "mean_iou":         model_means["mean_iou"]         > baseline_means["mean_iou"],
    "hausdorff":        model_means["hausdorff"]        < baseline_means["hausdorff"],
    "count_ratio":      abs(model_means["count_ratio"] - 1) < abs(baseline_means["count_ratio"] - 1),
    "vertex_reduction": True,   # any simplification is progress
}

comparison["Model better?"] = ["✓" if better[m] else "✗" for m in metrics_cols]

print("═" * 55)
print("MODEL vs BASELINE — Test set (Wageningen)")
print("═" * 55)
print(comparison.to_string())
print(f"\nOverall pixel accuracy: {overall_pixel_acc*100:.2f}%")

# Save
model_df.to_csv(CONFIG["output_root"] / "model_results.csv", index=False)
comparison.to_csv(CONFIG["output_root"] / "comparison_table.csv")
print("\nResults saved.")

## 7. Side-by-side metric comparison plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_specs = [
    ("mean_iou",    "IoU",                  True,  "higher is better"),
    ("hausdorff",   "Hausdorff dist. (m)",  False, "lower is better"),
    ("count_ratio", "Count ratio",          None,  "1.0 = perfect"),
]

for ax, (col, label, higher_better, note) in zip(axes, plot_specs):
    b_vals = baseline_df[col].dropna()
    m_vals = model_df[col].dropna()

    all_vals = pd.concat([b_vals, m_vals])
    bins = np.linspace(all_vals.min(), all_vals.max(), 25)

    ax.hist(b_vals, bins=bins, alpha=0.6, color="steelblue", label="Baseline (DP)", density=True)
    ax.hist(m_vals, bins=bins, alpha=0.6, color="tomato",    label="U-Net",         density=True)

    ax.axvline(b_vals.mean(), color="steelblue", linestyle="--", linewidth=1.5)
    ax.axvline(m_vals.mean(), color="tomato",    linestyle="--", linewidth=1.5)

    if col == "count_ratio":
        ax.axvline(1.0, color="black", linestyle=":", linewidth=1, label="ideal = 1.0")

    ax.set_title(f"{label}\n({note})")
    ax.set_xlabel(label)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.suptitle("Model vs Baseline — Test set metric distributions", fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "model_vs_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Visual comparison on a sample test tile

In [ ]:
from shapely.geometry import box

# Import baseline generalization (re-define inline)
def generalize_baseline_local(bgt, min_area=25.0, dp_tol=1.0):
    filtered = bgt[bgt.geometry.area >= min_area].copy()
    filtered["geometry"] = filtered.geometry.simplify(dp_tol, preserve_topology=True)
    filtered = filtered[filtered.geometry.is_valid & ~filtered.geometry.is_empty]
    return filtered

# Pick a tile near median model IoU
med = model_df["mean_iou"].median()
best_idx = (model_df["mean_iou"] - med).abs().idxmin()
example = test_tiles[best_idx]

bgt_ex = gpd.read_file(example["bgt"])
brt_ex = gpd.read_file(example["brt"])

# Baseline
base_ex = generalize_baseline_local(bgt_ex)

# Model prediction
x_arr = rasterize_bgt(bgt_ex, bgt_enc_col, n_in, CONFIG["image_size"])
x_t = torch.from_numpy(x_arr).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    pred_mask = model(x_t).argmax(dim=1).squeeze(0).cpu().numpy()
model_ex = mask_to_geodataframe(
    pred_mask, bgt_ex.total_bounds, brt_encoder,
    CONFIG["min_area_post_vectorize"], CONFIG["crs"]
)

# Plot
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
plots = [
    (bgt_ex,    "steelblue", f"BGT input\n({len(bgt_ex)} polygons)"),
    (base_ex,   "darkorange",f"Baseline (DP)\n({len(base_ex)} polygons)"),
    (model_ex,  "mediumpurple", f"U-Net prediction\n({len(model_ex)} polygons)"),
    (brt_ex,    "tomato",    f"BRT target\n({len(brt_ex)} polygons)"),
]
for ax, (gdf, color, title) in zip(axes, plots):
    if len(gdf) > 0:
        gdf.plot(ax=ax, color=color, edgecolor="white", linewidth=0.3, alpha=0.85)
    ax.set_title(title)
    ax.set_axis_off()

m = model_df.loc[best_idx]
plt.suptitle(
    f"Test tile — Model IoU: {m['mean_iou']:.3f} | "
    f"Baseline IoU: {baseline_df.iloc[best_idx]['mean_iou'] if best_idx < len(baseline_df) else 'N/A'}",
    fontsize=12
)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "test_tile_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Final summary

In [ ]:
print("═" * 60)
print("FINAL EVALUATION SUMMARY — Test set (Wageningen)")
print("═" * 60)
print(f"\n{'Metric':<25} {'Baseline':>12} {'U-Net':>12} {'Better?':>10}")
print("-" * 60)

for col, label, note in [
    ("mean_iou",         "Mean IoU",          "↑ better"),
    ("hausdorff",        "Hausdorff (m)",      "↓ better"),
    ("count_ratio",      "Count ratio",        "1.0 ideal"),
    ("vertex_reduction", "Vertex reduction",   "↑ better"),
]:
    bv = baseline_df[col].mean()
    mv = model_df[col].mean()
    if col == "mean_iou" or col == "vertex_reduction":
        flag = "✓" if mv > bv else "✗"
    elif col == "hausdorff":
        flag = "✓" if mv < bv else "✗"
    else:
        flag = "✓" if abs(mv - 1) < abs(bv - 1) else "✗"
    print(f"{label:<25} {bv:>12.4f} {mv:>12.4f} {flag:>10}  ({note})")

print("-" * 60)
print(f"{'Pixel accuracy':<25} {'—':>12} {overall_pixel_acc*100:>11.2f}%")
print("\nAll outputs saved to:", CONFIG["output_root"])